# Retrieval-Augmented Generation (RAG) Fundamentals

## What is RAG?
RAG combines **retrieval** (fetching relevant documents) with **generation** (LLM response) to produce accurate, grounded answers.

**Why RAG?**
- LLMs have a knowledge cutoff date
- LLMs can't access private/proprietary data
- LLMs hallucinate when uncertain
- Fine-tuning is expensive; RAG is cheaper to update

## The RAG Pipeline

```
INDEXING (offline):
  Documents → Load → Chunk → Embed → Store in VectorDB

QUERYING (online):
  Query → Embed → Retrieve → Augment Prompt → LLM → Answer
```

## Similarity Search

**Cosine Similarity** (most common):
$$\cos(\theta) = \frac{A \cdot B}{\|A\| \|B\|} = \frac{\sum_{i=1}^n A_i B_i}{\sqrt{\sum A_i^2} \cdot \sqrt{\sum B_i^2}}$$

**Dot Product** (fast, used when vectors are normalized):
$$A \cdot B = \sum_{i=1}^n A_i B_i$$

**Euclidean Distance** (L2):
$$d(A,B) = \sqrt{\sum_{i=1}^n (A_i - B_i)^2}$$

## Chunking Strategies

| Strategy | Description | Best For |
|----------|-------------|----------|
| Fixed size | Split every N characters | Simple text |
| Recursive character | Split by paragraph → sentence → word | General purpose |
| Semantic | Split by meaning change | Dense technical docs |
| Sentence | One sentence per chunk | Q&A systems |
| Markdown | Split by headers | Documentation |

In [1]:
# pip install langchain langchain-community langchain-openai faiss-cpu sentence-transformers pymupdf

# --- 1. Document Loading ---
from langchain_community.document_loaders import PyMuPDFLoader, WebBaseLoader, CSVLoader, JSONLoader
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# Load a PDF
# loader = PyMuPDFLoader("document.pdf")
# docs = loader.load()

# Load from web
# loader = WebBaseLoader("https://example.com/article")
# docs = loader.load()

# Simulate documents
from langchain_core.documents import Document

raw_docs = [
    Document(page_content="RAG stands for Retrieval-Augmented Generation. It combines retrieval systems with LLMs.", metadata={"source": "intro.txt"}),
    Document(page_content="Vector databases store embeddings and enable fast similarity search. Popular ones include FAISS, Chroma, Pinecone.", metadata={"source": "vectordb.txt"}),
    Document(page_content="Embeddings are dense vector representations of text. OpenAI's text-embedding-3-small produces 1536-dim vectors.", metadata={"source": "embeddings.txt"}),
    Document(page_content="Chunking is the process of splitting documents into smaller pieces before embedding them.", metadata={"source": "chunking.txt"}),
    Document(page_content="The retrieval step fetches the top-k most similar chunks to the user query from the vector store.", metadata={"source": "retrieval.txt"}),
]

print(f'Loaded {len(raw_docs)} documents')
print(f'First doc: {raw_docs[0].page_content[:100]}')

/tmp/ipykernel_163981/1618032362.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader, WebBaseLoader, CSVLoader, JSONLoader


USER_AGENT environment variable not set, consider setting it to identify your requests.


Loaded 5 documents
First doc: RAG stands for Retrieval-Augmented Generation. It combines retrieval systems with LLMs.


In [2]:
# --- 2. Text Splitting ---
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter

# Recursive character splitter (recommended default)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,  # overlap for context continuity
    separators=["\n\n", "\n", ". ", " ", ""]  # priority order
)

chunks = splitter.split_documents(raw_docs)
print(f'Split into {len(chunks)} chunks')
for i, chunk in enumerate(chunks[:3]):
    print(f'\nChunk {i}: [{len(chunk.page_content)} chars] {chunk.page_content[:80]}...')

Split into 5 chunks

Chunk 0: [87 chars] RAG stands for Retrieval-Augmented Generation. It combines retrieval systems wit...

Chunk 1: [114 chars] Vector databases store embeddings and enable fast similarity search. Popular one...

Chunk 2: [111 chars] Embeddings are dense vector representations of text. OpenAI's text-embedding-3-s...


In [3]:
# --- 3. Embeddings ---
import numpy as np

# Option A: Sentence Transformers (free, local)
# pip install sentence-transformers
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')  # 384-dim, fast
# Other good models:
# 'BAAI/bge-m3'          multilingual, 1024-dim
# 'intfloat/e5-large-v2' strong English, 1024-dim
# 'nomic-ai/nomic-embed-text-v1.5' 768-dim, good for RAG

texts = [chunk.page_content for chunk in chunks]
embeddings = model.encode(texts, show_progress_bar=False)
print(f'Embedding shape: {embeddings.shape}')  # (n_chunks, 384)

# Cosine similarity from scratch
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

query = "What is a vector database?"
query_emb = model.encode([query])[0]
sims = [cosine_similarity(query_emb, emb) for emb in embeddings]
top_idx = np.argsort(sims)[::-1][:2]
print(f'\nQuery: {query}')
for idx in top_idx:
    print(f'  Score {sims[idx]:.3f}: {texts[idx][:80]}')

Embedding shape: (5, 384)

Query: What is a vector database?
  Score 0.600: Vector databases store embeddings and enable fast similarity search. Popular one
  Score 0.463: The retrieval step fetches the top-k most similar chunks to the user query from 


In [4]:
# --- 4. Vector Store with FAISS ---
import faiss

class FAISSVectorStore:
    def __init__(self, dim):
        self.index = faiss.IndexFlatIP(dim)  # Inner Product (cosine if normalized)
        self.docs = []

    def add(self, texts, embeddings):
        # Normalize for cosine similarity
        norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
        normalized = embeddings / norms
        self.index.add(normalized.astype('float32'))
        self.docs.extend(texts)

    def search(self, query_emb, k=3):
        q = query_emb / np.linalg.norm(query_emb)
        q = q.reshape(1, -1).astype('float32')
        scores, indices = self.index.search(q, k)
        return [(self.docs[i], scores[0][j]) for j, i in enumerate(indices[0])]

store = FAISSVectorStore(dim=embeddings.shape[1])
store.add(texts, embeddings)

results = store.search(query_emb, k=3)
print('Top 3 results:')
for doc, score in results:
    print(f'  [{score:.3f}] {doc[:80]}')

Top 3 results:
  [0.600] Vector databases store embeddings and enable fast similarity search. Popular one
  [0.463] The retrieval step fetches the top-k most similar chunks to the user query from 
  [0.353] Embeddings are dense vector representations of text. OpenAI's text-embedding-3-s


In [5]:
# --- 5. Full RAG Pipeline with LangChain ---
# Using Chroma (persistent local vector store)
# pip install chromadb langchain-chroma

RAG_CODE = '''
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Vector store (persisted to disk)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

# Retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",  # or "mmr"
    search_kwargs={"k": 4}
)

# Prompt
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:

{context}

Question: {question}

If the answer is not in the context, say "I don't know."
""")

def format_docs(docs):
    return "\\n\\n".join(doc.page_content for doc in docs)

# RAG chain (LCEL)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | ChatOpenAI(model="gpt-4o-mini")
    | StrOutputParser()
)

# Ask a question
answer = rag_chain.invoke("What is RAG?")
print(answer)
'''
print('Full LangChain RAG pipeline:')
print(RAG_CODE)

Full LangChain RAG pipeline:

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Vector store (persisted to disk)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

# Retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",  # or "mmr"
    search_kwargs={"k": 4}
)

# Prompt
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:

{context}

Question: {question}

If the answer is not in the context, say "I don't know."
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# RAG chain (LCEL)
rag_chain = (
    {"context": retriever 

In [6]:
# --- 6. MMR (Maximal Marginal Relevance) Retrieval ---
# MMR balances relevance with diversity to avoid redundant retrieved chunks
# MMR score: lambda * similarity(doc, query) - (1-lambda) * max_similarity(doc, selected)

def mmr_retrieval(query_emb, doc_embeddings, docs, k=3, lambda_=0.5):
    """
    MMR: select documents that are relevant to query but diverse among themselves.
    lambda_=1 → pure relevance; lambda_=0 → pure diversity
    """
    # Normalize
    q = query_emb / np.linalg.norm(query_emb)
    D = doc_embeddings / np.linalg.norm(doc_embeddings, axis=1, keepdims=True)

    relevance = D @ q
    selected = []
    remaining = list(range(len(docs)))

    for _ in range(k):
        if not selected:
            best = int(np.argmax(relevance))
        else:
            sel_embs = D[selected]
            redundancy = np.max(D[remaining] @ sel_embs.T, axis=1)
            rel = relevance[remaining]
            scores = lambda_ * rel - (1 - lambda_) * redundancy
            best = remaining[int(np.argmax(scores))]
        selected.append(best)
        remaining.remove(best)

    return [(docs[i], relevance[i]) for i in selected]

mmr_results = mmr_retrieval(query_emb, embeddings, texts, k=3)
print('MMR results (diverse + relevant):')
for doc, score in mmr_results:
    print(f'  [{score:.3f}] {doc[:80]}')

MMR results (diverse + relevant):
  [0.600] Vector databases store embeddings and enable fast similarity search. Popular one
  [0.220] RAG stands for Retrieval-Augmented Generation. It combines retrieval systems wit
  [0.256] Chunking is the process of splitting documents into smaller pieces before embedd


## Additional Learning Resources

### Papers
- [Retrieval-Augmented Generation (original paper)](https://arxiv.org/abs/2005.11401) Lewis et al., 2020
- [RAG Survey](https://arxiv.org/abs/2312.10997) Gao et al., 2023

### Documentation
- [LangChain RAG Tutorial](https://python.langchain.com/docs/tutorials/rag/)
- [LlamaIndex RAG Docs](https://docs.llamaindex.ai/en/stable/understanding/rag/)
- [FAISS Documentation](https://faiss.ai/)
- [Chroma Docs](https://docs.trychroma.com/)
- [Pinecone Learning Center](https://www.pinecone.io/learn/series/rag/)

### Videos & Courses
- [RAG from Scratch LangChain YouTube](https://www.youtube.com/playlist?list=PLfaIDFEXuae2LXb3FO0Q9KxxmNa1efnKi)
- [Deeplearning.ai RAG Course](https://www.deeplearning.ai/short-courses/building-and-evaluating-advanced-rag/)

### Embedding Model Rankings
- [MTEB Leaderboard](https://huggingface.co/spaces/mteb/leaderboard)